# 🏀 The Dream Team — Picking 12 Players With Integer Optimization

**ISOM 839 · Session 7 · Extra Practice · Prof. Hasan Arslan · Suffolk University**

It is the spring of 2028. The Olympics are in Los Angeles, and the USA Basketball selection committee has narrowed the field to **20 players**. They need **12**.

Every fan has an opinion. The committee has **rules**: enough point guards, enough big men, at least two college players, a minimum average for rebounds, assists, height and defense, no more than three players from one franchise. And, because these are stars, two personal conditions: one player refuses to play if another is picked, and two longtime teammates come as a pair or not at all.

The goal: **the highest average scoring** a 12-player roster can have while following every rule.

This one problem uses almost everything from Sessions 6 and 7: yes/no variables, **covering** constraints (positions), **packing** (team size, franchise cap), **if-then** logic, a **pair** rule, and one new trick: turning an *average* into a linear constraint.

> *Adapted from the "National Basketball Dream Team" case in Bertsimas & Freund, **Data, Models, and Decisions**, Ch. 9. The players, teams and numbers here are new and fictional.*

In [ ]:
%pip install -q gurobipy pandas matplotlib ortools
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
print('Ready')

---
## 1 · Meet the 20 finalists

Positions are separated by `/` when a player can play two of them. A player like Kofi Mensah (`PG/SG`) counts toward **both** position minimums if he makes the team. That is exactly why this is a *covering* problem and not a simple quota.

Height is in inches (79 inches = 6'7"). Defense is the scouts' rating on a 0–10 scale.

In [ ]:
rows = [
# id  name                positions league  team        reb ast pts height defense
( 1, 'Marcus Bell',       'PG',     'NBA',  'Celtics',    4, 10, 22, 74,  8),
( 2, 'Jalen Ortiz',       'PG',     'NBA',  'Nuggets',    5, 11, 19, 75,  9),
( 3, 'Devin Price',       'PG',     'NBA',  'Knicks',     3,  9, 26, 73,  6),
( 4, 'Kofi Mensah',       'PG/SG',  'NCAA', 'Duke',       4,  7, 18, 76,  7),
( 5, 'Tyrese Holloway',   'PG/SG',  'NBA',  'Mavericks',  6,  9, 27, 77,  7),
( 6, 'Andre Whitfield',   'SG',     'NBA',  'Celtics',    5,  5, 25, 78,  9),
( 7, 'Cameron Reyes',     'SG',     'NBA',  'Suns',       4,  4, 29, 77,  6),
( 8, 'Luka Brandt',       'SG',     'NCAA', 'Gonzaga',    6,  4, 16, 79,  8),
( 9, 'Isaiah Grant',      'SG/F',   'NBA',  'Bucks',      7,  5, 24, 80,  9),
(10, 'Noah Castellanos',  'SG/F',   'NBA',  'Celtics',    6,  6, 23, 79,  8),
(11, 'Elijah Moore',      'SG/F',   'NBA',  'Thunder',    8,  6, 21, 80, 10),
(12, 'Darius Fleming',    'F',      'NBA',  'Lakers',     9,  4, 28, 81,  8),
(13, 'Zion Okafor',       'F',      'NBA',  'Pelicans',  10,  3, 25, 82,  7),
(14, 'Rafael Duarte',     'F',      'NBA',  'Celtics',    8,  4, 27, 81,  9),
(15, 'Owen Kowalski',     'F',      'NCAA', 'UConn',      9,  2, 17, 82,  8),
(16, 'Trevor Lang',       'F/C',    'NBA',  'Heat',      11,  3, 19, 83,  9),
(17, 'Malik Adeyemi',     'C',      'NBA',  '76ers',     13,  4, 26, 84,  8),
(18, 'Victor Hale',       'C',      'NBA',  'Spurs',     12,  3, 22, 87, 10),
(19, 'Brandon Kerr',      'C',      'NBA',  'Nuggets',   11,  2, 18, 84,  9),
(20, 'Samuel Achebe',     'C',      'NCAA', 'Kentucky',  12,  1, 14, 85,  9),
]
players = pd.DataFrame(rows, columns=['id', 'name', 'positions', 'league', 'team',
                                      'reb', 'ast', 'pts', 'height', 'defense']).set_index('id')
P = list(players.index)

def plays(i, pos):
    return pos in players.positions[i].split('/')

players

---
## 2 · The committee's rules — and the pattern behind each one

| # | Rule | Pattern | As a constraint ( $x_i = 1$ if player $i$ is picked ) |
|---|---|---|---|
| 1 | Exactly 12 players | packing / equality | $\sum_i x_i = 12$ |
| 2 | At least 3 PG, 4 SG, 4 F, 3 C | **covering** | $\sum_{i \text{ plays PG}} x_i \ge 3$, and so on |
| 3 | At least 2 college (NCAA) players | covering | $\sum_{i \in \text{NCAA}} x_i \ge 2$ |
| 4 | Average rebounds ≥ 7, assists ≥ 5, height ≥ 79", defense ≥ 8.3 | **the average trick** (below) | $\sum_i \text{reb}_i\, x_i \ge 7 \times 12$ |
| 5 | Tyrese Holloway (5) will not play if Isaiah Grant (9) is picked | **if-then** | $x_5 + x_9 \le 1$ |
| 6 | Jalen Ortiz (2) and Brandon Kerr (19) come as a pair | **pair** | $x_2 = x_{19}$ |
| 7 | At most 3 players from one franchise | packing | $\sum_{i \in \text{Celtics}} x_i \le 3$ |

### The average trick
"The average rebounds of the team must be at least 7" looks nonlinear — an average is a sum divided by a count, and the count depends on who you pick. But the count is **fixed at 12** by rule 1. So

$$\frac{\sum_i \text{reb}_i\, x_i}{12} \ge 7 \quad\Longleftrightarrow\quad \sum_i \text{reb}_i\, x_i \ge 84$$

which is perfectly linear. Keep this trick; you will use it in the midterm case. (If the team size were *not* fixed, you would need a different trick: $\sum_i (\text{reb}_i - 7)\, x_i \ge 0$. Try proving to yourself that this works for any team size.)

**Rule 5 as a sentence check:** "if 9 is picked then 5 is not" means $x_9 = 1 \Rightarrow x_5 = 0$. The only forbidden combination is *both*, so $x_5 + x_9 \le 1$. It is the clopen rule from tonight's lab, wearing a jersey.

---
## 3 · The fan's roster: just take the 12 best scorers

Before building a model, try what most people would do. The `check` function below tests any roster against every rule. It is also how you should audit your own model's answer.

In [ ]:
STANDARDS = {'reb': 7, 'ast': 5, 'height': 79, 'defense': 8.3}
POS_MIN   = {'PG': 3, 'SG': 4, 'F': 4, 'C': 3}

def check(roster, standards=STANDARDS, show=True):
    t = players.loc[roster]
    broken = []
    if len(roster) != 12: broken.append(f'{len(roster)} players, not 12')
    for pos, k in POS_MIN.items():
        n = sum(plays(i, pos) for i in roster)
        if n < k: broken.append(f'only {n} {pos} (need {k})')
    if (t.league == 'NCAA').sum() < 2: broken.append(f'only {(t.league == "NCAA").sum()} NCAA players (need 2)')
    for stat, floor in standards.items():
        if t[stat].mean() < floor - 1e-9: broken.append(f'average {stat} {t[stat].mean():.2f} (need {floor})')
    if 5 in roster and 9 in roster: broken.append('Holloway (5) and Grant (9) both picked')
    if (2 in roster) != (19 in roster): broken.append('Ortiz (2) and Kerr (19) split up')
    for team, n in t.team.value_counts().items():
        if n > 3: broken.append(f'{n} players from the {team} (max 3)')
    if show:
        print(f'Average points: {t.pts.mean():.2f}')
        print('✅ Every rule satisfied' if not broken else '❌ Broken rules:\n   - ' + '\n   - '.join(broken))
    return broken

fans_pick = list(players.sort_values('pts', ascending=False).index[:12])
print('Fan roster:', ', '.join(players.name[i] for i in fans_pick), '\n')
check(fans_pick);

Great scoring, five broken rules: too few centers, no college players, soft on defense, the feud, and four Celtics. The committee would never sign it. Now the question is: **how many points per game does following the rules cost**, and which players does it take?

Fixing it by hand is a puzzle with $\binom{20}{12} = 125{,}970$ possible rosters. That is small enough to brute force, but add 10 more candidates and it is 86 million. Let the solver do it.

---
## 4 · The model

One yes/no variable per player. Everything else is a sum. The `build` function takes the standards and an optional rule to switch off — we will use that later to price each rule.

In [ ]:
def build(standards=STANDARDS, drop=None, relax=False, quiet=True):
    m = gp.Model('dream_team')
    m.Params.OutputFlag = 0 if quiet else 1
    x = m.addVars(P, vtype=GRB.CONTINUOUS if relax else GRB.BINARY, ub=1, name='pick')
    rules = {}

    rules['team size']   = m.addConstr(x.sum() == 12)
    for pos, k in POS_MIN.items():                                            # covering
        rules[f'{pos} minimum'] = m.addConstr(gp.quicksum(x[i] for i in P if plays(i, pos)) >= k)
    rules['2 NCAA players'] = m.addConstr(gp.quicksum(x[i] for i in P if players.league[i] == 'NCAA') >= 2)
    for stat, floor in standards.items():                                     # the average trick
        rules[f'avg {stat}'] = m.addConstr(gp.quicksum(players[stat][i] * x[i] for i in P) >= floor * 12)
    rules['Holloway/Grant feud'] = m.addConstr(x[5] + x[9] <= 1)              # if-then
    rules['Ortiz–Kerr pair']     = m.addConstr(x[2] == x[19])                 # pair
    for team in players.team.unique():                                        # franchise cap
        members = [i for i in P if players.team[i] == team]
        if len(members) > 3:
            rules[f'max 3 {team}'] = m.addConstr(gp.quicksum(x[i] for i in members) <= 3)

    if drop:
        m.remove(rules.pop(drop))
    m.setObjective(gp.quicksum(players.pts[i] * x[i] for i in P) / 12, GRB.MAXIMIZE)
    return m, x, rules

m, x, rules = build(quiet=False)
m.optimize()
team = [i for i in P if x[i].X > 0.5]
best = m.ObjVal

Read the log the way you learned last week: the LP relaxation bound, the incumbent, the MIP gap closing to 0%. For 20 binary variables it takes a blink. Now print the roster the way a coach would want to see it.

In [ ]:
def show_roster(roster):
    t = players.loc[roster].copy()
    t['picked'] = '✔'
    order = t.positions.str.split('/').str[0].map({'PG': 0, 'SG': 1, 'F': 2, 'C': 3})
    print(t.assign(o=order).sort_values(['o', 'pts'], ascending=[True, False])
           [['name', 'positions', 'league', 'team', 'pts', 'reb', 'ast', 'height', 'defense']].to_string())
    print()
    print('Team averages:', {s: round(float(t[s].mean()), 2) for s in ['pts', 'reb', 'ast', 'height', 'defense']})

show_roster(team)
print()
check(team)
print(f'\nFollowing the rules costs {players.loc[fans_pick].pts.mean() - best:.2f} points per game vs. the fan roster.')
left_out = [players.name[i] for i in fans_pick if i not in team]
print('Top-12 scorers who did NOT make it:', ', '.join(left_out))

### 💬 Discussion
Look at who got left off and who got on. A 14-point scorer made the team over a 29-point scorer. Is that a bad decision, or is it the price of a rule the committee chose? The next section answers that in points.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
picked = players.index.isin(team)
ax.scatter(players.defense[~picked], players.pts[~picked], s=110, c='#cccccc', edgecolor='#888', label='Left off')
ax.scatter(players.defense[picked],  players.pts[picked],  s=140, c='#f2b135', edgecolor='#7a4f00', label='Picked')
seen = {}
for i in P:
    spot = (players.defense[i], players.pts[i])
    k = seen.get(spot, 0); seen[spot] = k + 1          # two players with identical stats: stack labels
    ax.annotate(players.name[i].split()[1], spot,
                textcoords='offset points', xytext=(7, 4 - 11 * k), fontsize=8.5)
ax.set_xlabel('Defense rating (0–10)'); ax.set_ylabel('Points per game')
ax.set_title('The optimal roster is not simply the top-right corner')
ax.legend(loc='lower left'); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

---
## 5 · What does each rule cost?

This is the most useful question you can ask of any constrained model, and the one the committee chair will care about most. For each rule: switch it off, re-solve, and see how many points per game the team gains.

- **Gain > 0:** the rule is *binding*. It is costing you scoring, and the number is its price.
- **Gain = 0:** the rule is free *at this optimum*. Everyone argued about it, and it did not matter.

This is the integer-programming cousin of the shadow prices from Session 3. (Integer programs do not give you shadow prices for free, so you compute them this way.)

In [ ]:
prices = []
for rule in rules:
    if rule == 'team size':
        continue
    m2, x2, _ = build(drop=rule)
    m2.optimize()
    new = [i for i in P if x2[i].X > 0.5]
    prices.append({'rule': rule, 'gain if dropped (pts/game)': round(m2.ObjVal - best, 3),
                   'comes in': ', '.join(players.name[i].split()[1] for i in new if i not in team) or '—',
                   'goes out': ', '.join(players.name[i].split()[1] for i in team if i not in new) or '—'})
price_table = pd.DataFrame(prices).sort_values('gain if dropped (pts/game)', ascending=False)
pd.set_option('display.max_colwidth', None)
price_table

In [ ]:
binding = price_table[price_table['gain if dropped (pts/game)'] > 1e-6]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(binding.rule[::-1], binding['gain if dropped (pts/game)'][::-1], color='#e08e3c')
ax.set_xlabel('Points per game the rule costs'); ax.set_title('The price of each binding rule')
ax.grid(axis='x', alpha=0.25); plt.tight_layout(); plt.show()

### 💬 Discussion
- The **two-college-players rule** is the most expensive single rule. Is it worth almost a point a game? That is not an optimization question. It is a values question — and now the committee can argue about it with a number instead of a feeling.
- The **max-3-Celtics rule** broke the fan roster, yet it costs nothing at the optimum. Why? (Hint: look at which Celtic the model left off, and what else he would have to replace.)
- These prices are **not additive.** Drop two rules at once and the gain can be more, or less, than the sum. Try it.

---
## 6 · Why not just let the LP decide and round?

Change one word — `BINARY` to `CONTINUOUS` — and solve again.

In [ ]:
mr, xr, _ = build(relax=True)
mr.optimize()
print(f'LP relaxation: {mr.ObjVal:.3f} points per game   (integer optimum: {best:.3f})\n')
for i in P:
    if 1e-6 < xr[i].X < 1 - 1e-6:
        print(f'  pick {xr[i].X:.0%} of {players.name[i]}')

The LP happily picks a *fraction* of a player. Look where two of the fractions land: 10% of Ortiz and 10% of Kerr, exactly honoring the pair rule in the silliest possible way. Round them and you either break the pair or break the team size. The LP value is still useful: it is the **upper bound** branch & bound starts from. The gap between the two numbers is the price of reality.

---
## 7 · Exercises

### Exercise 1 — The committee raises the bar
The chair wants a tougher team: **average rebounds ≥ 8 and average assists ≥ 6**, everything else unchanged.

1. Solve with both changes. What happens?
2. Now try each change **alone**. Which one is free, and which one costs points? Who comes on and off the roster?
3. Write two sentences to the chair explaining what you found. No jargon.

In [ ]:
tough = {**STANDARDS, 'reb': 8, 'ast': 6}
# TODO: build(standards=tough), optimize, and check m.Status
#       GRB.OPTIMAL == 2, GRB.INFEASIBLE == 3
# TODO: then try {**STANDARDS, 'reb': 8} and {**STANDARDS, 'ast': 6} separately

### Exercise 2 — The injury report (worked for you)
An optimal plan can be **brittle**. Suppose one of the 12 gets hurt the week before the Games, and the committee wants to swap in exactly one replacement while keeping the other 11. Does a legal one-for-one swap always exist?

In [ ]:
def best_swap(injured):
    others = [i for i in team if i != injured]
    m3, x3, _ = build()
    for i in others: m3.addConstr(x3[i] == 1)     # keep the other 11
    m3.addConstr(x3[injured] == 0)                # the injured player is out
    m3.optimize()
    if m3.Status != GRB.OPTIMAL:
        return None, None
    sub = [i for i in P if x3[i].X > 0.5 and i not in team][0]
    return sub, m3.ObjVal

for i in team:
    sub, val = best_swap(i)
    who = players.name[i]
    print(f'{who:18s} injured → ' + (f'bring in {players.name[sub]:16s} ({val:.2f} pts/game)' if sub else '❌ no legal one-for-one swap'))

That is the lesson most textbooks skip: **the best plan for the expected case is often the most fragile one.** The optimizer pushed every rule right to its edge, so losing almost anyone breaks something.

### Exercise 3 — Choose two alternates (open-ended)
The committee will also name **two alternates**. Propose a method for choosing them, then implement it. Some directions to consider:

- **Coverage:** pick the two alternates that make the most injuries fixable by a one-for-one swap. Can two alternates cover all twelve? If not, what is the best you can do?
- **Robust roster:** change the *starting 12* so that every single injury has a legal swap from the alternates. How many points per game does that robustness cost? (This is a bigger model: one copy of the roster for each injury scenario. It is the same idea as ECCO's *stochastic* MIP — plan for the scenarios, not just the forecast.)
- Whatever you choose, state the trade-off in one sentence the chair would understand.

In [ ]:
# TODO: your alternates method here

### Exercise 4 — The same model in CP-SAT
Rebuild the base model in Google's CP-SAT and confirm you get the same average. Two things to watch:

- CP-SAT only accepts **integer** coefficients. The defense floor is $8.3 \times 12 = 99.6$. Since the team's total defense is a whole number, $\ge 99.6$ means the same as $\ge 100$.
- Maximize **total** points (an integer) instead of the average, and divide by 12 when you print.

In [ ]:
from ortools.sat.python import cp_model

cp = cp_model.CpModel()
y = {i: cp.NewBoolVar(f'pick_{i}') for i in P}
cp.Add(sum(y.values()) == 12)
# TODO: positions, NCAA, the four averages (integers!), feud, pair, franchise cap
# With only the team-size rule you will see the fan roster's 25.33. Add the rest and it should drop to the Gurobi optimum.
cp.Maximize(sum(int(players.pts[i]) * y[i] for i in P))
solver = cp_model.CpSolver()
status = solver.Solve(cp)
print(solver.StatusName(status), '· average points', solver.ObjectiveValue() / 12)

### Exercise 5 (stretch) — A different objective
The chair's co-chair is a defense-first coach. **Maximize average defense** subject to every rule **plus** average points ≥ 22. Which players swap in? Plot points against defense for the two rosters. Which one would you send to Los Angeles, and why?

In [ ]:
# TODO: copy build(), change the objective, add the points floor

---
## What to take into the midterm case

1. **Read every rule and name its pattern** — covering (≥), packing (≤), if-then, pair — before you type a line of code.
2. **Averages become sums** when the count is fixed. Beacon's fill rates and service levels work the same way.
3. **Price your rules.** Switch one off, re-solve, report the gain in dollars. That table is half of a great COO memo.
4. **Audit the answer** with a `check()` function. A model that solves is not the same as a model that is right.
5. **Optimal is fragile.** Ask what happens when one assumption breaks. On Nov 4 the VP will ask about the cold-snap forecast being 20% off. Have an answer.